In [1]:
import sys
# here, either reference to satclip repo goes, or can try and create git submodule 🤷‍♀️ 
# https://github.com/microsoft/satclip/tree/main
# sys.path.append('/home/cbutsko/Desktop/cbutsko_experiments/satclip/satclip')

sys.path.append('/home/cbutsko/Desktop/worldcereal-classification/notebooks/')

# general
import numpy as np
import pandas as pd 
import geopandas as gpd
import itertools
import gc
import re
import torch
import glob
import json
import random
import rioxarray
import xarray as xr
import rasterio as rio
import os
import swifter
from h3 import h3
import h3pandas

# modeling
from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, precision_score, recall_score
from sklearn.model_selection import train_test_split
# from sklearn.utils.validation import check_is_fitted, check_array
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from scipy.signal import find_peaks
# hierarchical clsfc libs
from hiclass import LocalClassifierPerParentNode, LocalClassifierPerNode
from cbutsko_utils import CatBoostClassifierWrapper, LocalClassifierPerNodeWrapper, LocalClassifierPerParentNodeWrapper

# plotting and output
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from cbutsko_utils import plot_confusion_matrix
import warnings
warnings.filterwarnings(action='ignore')
tqdm.pandas()

# SatClip libs
# from load import get_satclip
# from cbutsko_utils import prepare_satclip_embeddings

# processing utils
from cbutsko_utils import patch2feats, process_raw_features_input_df

# hierarchical clsfc libs
from hiclass import LocalClassifierPerParentNode, LocalClassifierPerNode
from cbutsko_utils import CatBoostClassifierWrapper, LocalClassifierPerNodeWrapper, LocalClassifierPerParentNodeWrapper

In [2]:
# read files with mappings to crop names and hierarchy

label_columns = ['cropland', 'landcover', 'cropgroup', 'croptype', 'worldcereal_croptype']

wc2ec_map = pd.read_csv('/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/wc2eurocrops_map.csv')
wc2ec_map['ec_code'] = wc2ec_map['ec_code'].str.replace('-','').astype(int)

ec_map = pd.read_csv('/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/eurocrops_map_wcr_edition.csv',)
ec_map = ec_map[ec_map['ec_code'].notna()]
ec_map['ec_code'] = ec_map['ec_code'].str.replace('-','').astype(int)
ec_map = ec_map.apply(lambda x: x[:x.last_valid_index()].ffill(), axis=1)
ec_map.set_index('ec_code', inplace=True)

code2name_map = ec_map[['{}_label'.format(label_columns[-1]),'{}_name'.format(label_columns[-1])]]
code2name_map = code2name_map[code2name_map['{}_label'.format(label_columns[-1])]>0].drop_duplicates()
code2name_map = dict(code2name_map.to_numpy())

tfpath = '/home/cbutsko/Desktop/worldcereal-referencedata/harmonization/resources/ewoc_legend_sampling_edition.csv'
sampling_crop_legend = pd.read_csv(tfpath, index_col='ewoc_code')

not_crop_classes = [
    'pasture_meadows',
    'trees_unspecified',
    'non_cropland_incl_perennial',
    'built_up',
    'shrubland',
    'open_water',
    'other_permanent_crops',
    'fruit_nuts',
    'herbaceous_vegetation',
    'permanent_crops',
    'non_cropland_excl_perennial',
    'greenhouse_foil_film_indoor',
    'bare_sparsely_vegetated',
    'kitchen_allotment_gardens',
    'not_cultivated_fallow'
    ]

dataset2refid_map_fpath = '/home/cbutsko/Desktop/worldcereal-referencedata/harmonization/resources/dataset2refid_map.json'
with open(dataset2refid_map_fpath) as f:
    dataset2refid_map = json.load(f)
dataset2refid_map = {value: key for key, value in dataset2refid_map.items()}

realms_fpath = '/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/realms.gpkg'
realms = gpd.read_file(realms_fpath)

realm_names = {
    0:'Unused',
    1:'Indian Islands',
    2:'Australia',
    3:'Europe',
    4:'North America',
    5:'Africa',
    6:'South America',
    7:'Asia',
    8:'Pacific Islands',
}

wc_modest_classes = {
    110101000:'wheat',
    110102000:'barley',
    110103000:'rye',
    110106000:'maize',
    110107000:'millet_sorghum',
    110600001:'sunflower',
    110600003:'rapeseed_rape',
    110600002:'soy_soybeans',
    119999999:'other_crop',
}
# with open('resources/croptype_multiclass_class2name_map.json', 'w') as f:
#     json.dump(wc_modest_classes, f, indent=4)

In [3]:
# read parquet files, assign hierarchical labels and do small clean-ups 
tpath = '/vitodata/worldcereal/features/benchmarking_features/calval/'
# tpath = '/vitodata/worldcereal/features/benchmarking_features/calval/expertfts-10d_calval.parquet'
# freq = '10d'
freq = 'monthly'

data_df = process_raw_features_input_df(
    '{}rawts-{}_calval.parquet'.format(tpath, freq),
    # tpath,
    wc2ec_map,
    ec_map,
    label_columns,
    add_countries=True
    )

data_df = data_df[data_df['country_code'].notna()]
# data_df['h3_l3_cell'] = data_df.swifter.progress_bar(False).apply(lambda xx: h3.geo_to_h3(
#     xx['lat'],
#     xx['lon'],
#     resolution=3
#     ), axis=1)
# gc.collect()

# best_cropland_model_preds = pd.read_csv('/home/cbutsko/Desktop/cbutsko_experiments/country_stratified_split_finetuned_presto.csv')
# best_cropland_model_preds.set_index('sample_id', inplace=True)
# data_df['crop_prob'] = data_df.index.map(best_cropland_model_preds['crop_prob'])
data_df['sampling_crop_name'] = data_df['ec_code'].map(sampling_crop_legend['sampling_croptype_name'])
data_df = data_df[data_df['sampling_crop_name']!='cropland_unspecified']
data_df['dataset_name'] = data_df['ref_id'].map(dataset2refid_map)

data_df['worldcereal_croptype_small'] = data_df['worldcereal_croptype']
data_df.loc[~data_df['worldcereal_croptype_small'].isin(wc_modest_classes.keys()),'worldcereal_croptype_small'] = 119999999

In [ ]:
croptype_spatial_generalization_test_split_fpath = 'resources/croptype_spatial_generalization_test_split_samples.csv'
croptype_spatial_generalization_test_split_samples = pd.read_csv(croptype_spatial_generalization_test_split_fpath)

In [ ]:
data_df[data_df.index.isin(croptype_spatial_generalization_test_split_samples['sample_id'])]['worldcereal_croptype_small'].map(wc_modest_classes).value_counts()

In [ ]:
data_df['worldcereal_croptype_small'].value_counts()

In [ ]:
croptype_multiclass_map = data_df[['CROPTYPE_LABEL','worldcereal_croptype_small']].value_counts().reset_index()
croptype_multiclass_map = croptype_multiclass_map[['CROPTYPE_LABEL','worldcereal_croptype_small']]
croptype_multiclass_map['CROPTYPE_LABEL'] = croptype_multiclass_map['CROPTYPE_LABEL'].astype(int)
croptype_multiclass_map.set_index('CROPTYPE_LABEL', inplace=True)
croptype_multiclass_map = croptype_multiclass_map.to_dict()['worldcereal_croptype_small']
with open('resources/croptype_multiclass_map.json', 'w') as f:
    json.dump(croptype_multiclass_map, f, indent=4)

In [ ]:
croptype_multiclass_map.keys()

In [ ]:
data_df = gpd.GeoDataFrame(
    data_df, 
    geometry=gpd.points_from_xy(data_df['lon'], data_df['lat']), 
    crs='EPSG:4326')

In [ ]:
data_df = data_df.sjoin(realms, how='inner', predicate='within')

### Density plots by continent

In [ ]:
countries = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
countries_h3 = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
france_borders = gpd.read_file('/home/cbutsko/Desktop/cbutsko_experiments/france_euro_borders.gpkg')
norway_borders = gpd.read_file('/home/cbutsko/Desktop/cbutsko_experiments/norway_euro_borders_trimmed.gpkg')

countries_h3 = countries_h3.h3.polyfill(3, explode=True)
countries_h3 = countries_h3[countries_h3['h3_polyfill'].notna()]
countries_h3 = countries_h3.set_index('h3_polyfill').h3.h3_to_geo_boundary()

countries_to_remove = ['Greenland','Iceland']
countries = countries[~countries['name'].isin(countries_to_remove)]
countries_h3 = countries_h3[~countries_h3['name'].isin(countries_to_remove)]

countries.loc[countries['name']=='France','geometry'] = france_borders['geometry'].iloc[0]
countries_h3.loc[countries_h3['name']=='France','geometry'] = france_borders['geometry'].iloc[0]
countries.loc[countries['name']=='Norway','geometry'] = norway_borders['geometry'].iloc[0]
countries_h3.loc[countries_h3['name']=='Norway','geometry'] = norway_borders['geometry'].iloc[0]

sample_counts = data_df['h3_l3_cell'].value_counts()
# sample_counts = data_df[
#     data_df['cropland_wc']==0
# ]['h3_l3_cell'].value_counts()
countries_h3['n_samples'] = countries_h3.index.map(sample_counts)

fig, axs = plt.subplots(
    ncols=2, 
    nrows=3, 
    figsize=(8,16),
    )
fig.tight_layout()
gs = axs[2, 1].get_gridspec()
for ax in axs[-1, -2:]:
    ax.remove()
axbig = fig.add_subplot(gs[-1,-2:])
fig.suptitle('WorldCereal Phase I Samples Distribution', y=1.03)
for ii,continent in enumerate(['Africa','North America','Europe','South America','Asia']):
    if continent=='Europe':
        continent_df = countries[
            (countries['continent']==continent) & 
            (countries['name']!='Russia')
            ]
        continent_h3_df = countries_h3[
            (countries_h3['continent']==continent) & 
            (countries_h3['name']!='Russia')
            ]
        continent_df.plot(
            color='lightgrey',
            ax=axs.flatten()[ii])
        continent_h3_df.plot(
            column='n_samples', 
            cmap='Greens',
            ax=axs.flatten()[ii]
            )
        axs.flatten()[ii].xaxis.set_ticklabels([])
        axs.flatten()[ii].xaxis.set_ticks([])
        axs.flatten()[ii].yaxis.set_ticklabels([])
        axs.flatten()[ii].yaxis.set_ticks([])
        axs.flatten()[ii].title.set_text("""
        {}
        Total samples: {}
        """.format(continent, int(continent_h3_df['n_samples'].sum())))
    elif continent=='Asia':
        continent_df = countries[countries['continent']==continent]
        continent_h3_df = countries_h3[countries_h3['continent']==continent]
        continent_df.plot(
            color='lightgrey',
            ax=axbig)
        continent_h3_df.plot(
            column='n_samples', 
            cmap='Greens',
            ax=axbig)
        axbig.xaxis.set_ticklabels([])
        axbig.xaxis.set_ticks([])
        axbig.yaxis.set_ticklabels([])
        axbig.yaxis.set_ticks([])
        axbig.title.set_text("""
        {}
        Total samples: {}
        """.format(continent, int(continent_h3_df['n_samples'].sum())))
    elif continent=='Russia':
        continent_df = countries[(countries['name']=='Russia')]
        continent_h3_df = countries_h3[(countries_h3['name']=='Russia')]
    else:
        continent_df = countries[countries['continent']==continent]
        continent_h3_df = countries_h3[countries_h3['continent']==continent]
        continent_df.plot(
            color='lightgrey',
            ax=axs.flatten()[ii])
        continent_h3_df.plot(
            column='n_samples', 
            cmap='Greens',
            ax=axs.flatten()[ii],
            # legend=True
            )
        axs.flatten()[ii].xaxis.set_ticklabels([])
        axs.flatten()[ii].xaxis.set_ticks([])
        axs.flatten()[ii].yaxis.set_ticklabels([])
        axs.flatten()[ii].yaxis.set_ticks([])
        axs.flatten()[ii].title.set_text("""
        {}
        Total samples: {}
        """.format(continent, int(continent_h3_df['n_samples'].sum())))
plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/djf_samples_distribution_phase1_continents.png', dpi=300)

### Density plots, comparison with new sampling procedure

In [ ]:
processed_datasets_dir = '/vitodata/worldcereal/data/RDM'
processed_datasets_fnames = glob.glob('{}/**/*.geoparquet'.format(processed_datasets_dir), recursive=True)
processed_datasets_fnames = [xx for xx in processed_datasets_fnames if ('_part' not in xx) and ('FRA' not in xx) and ('2021_AUT' not in xx) and ('2021_LTU' not in xx)]
processed_datasets = [xx.split('/')[-1].split('.')[0] for xx in processed_datasets_fnames]
processed_datasets = [xx for xx in processed_datasets if xx in data_df['dataset_name'].unique()]

In [ ]:
world_bounds_vector = gpd.read_file('/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/world-administrative-boundaries/world-administrative-boundaries.shp')
world_bounds_raster = rio.open('/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/world-administrative-boundaries/world-administrative-boundaries.tif')

In [ ]:
tres = []
pbar = tqdm(total=len(processed_datasets_fnames))
not_processed_lst = []
for dataset_path in processed_datasets_fnames:
    pbar.update(1)

    gdf = gpd.read_parquet(dataset_path, columns=['ewoc_code','extract','h3_l3_cell','geometry'])
    gdf = gdf[gdf['extract']>0]
    gdf['lon'] = gdf['geometry'].centroid.x
    gdf['lat'] = gdf['geometry'].centroid.y

    _country_inds = [xx[0] for xx in world_bounds_raster.sample(gdf[['lon','lat']].values)]
    gdf['country_code'] = [world_bounds_vector['color_code'].iloc[int(xx-1)] if xx>0 else 'None' for xx in _country_inds]

    gdf = gdf[['ewoc_code','extract','country_code','h3_l3_cell']]
    gc.collect()
    dataset_name = dataset_path.split('/')[-1].split('.')[0]
    gdf['dataset_name'] = dataset_name
    tres.append(gdf)

phase2_data_df = pd.concat(tres)
# phase2_data_df['crop_name'] = phase2_data_df['ewoc_code'].map(ec_map.set_index('Code')['Crop_type'])
phase2_data_df['sampling_crop_name'] = phase2_data_df['ewoc_code'].map(sampling_crop_legend['sampling_croptype_name'])
phase2_data_df = phase2_data_df[phase2_data_df['sampling_crop_name']!='cropland_unspecified']

In [ ]:
countries = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
countries_h3 = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
france_borders = gpd.read_file('/home/cbutsko/Desktop/cbutsko_experiments/france_euro_borders.gpkg')
norway_borders = gpd.read_file('/home/cbutsko/Desktop/cbutsko_experiments/norway_euro_borders_trimmed.gpkg')

countries_h3 = countries_h3.h3.polyfill(3, explode=True)
countries_h3 = countries_h3[countries_h3['h3_polyfill'].notna()]
countries_h3 = countries_h3.set_index('h3_polyfill').h3.h3_to_geo_boundary()

countries_to_remove = ['Greenland','Iceland']
countries = countries[~countries['name'].isin(countries_to_remove)]
countries_h3 = countries_h3[~countries_h3['name'].isin(countries_to_remove)]

countries.loc[countries['name']=='France','geometry'] = france_borders['geometry'].iloc[0]
countries_h3.loc[countries_h3['name']=='France','geometry'] = france_borders['geometry'].iloc[0]
countries.loc[countries['name']=='Norway','geometry'] = norway_borders['geometry'].iloc[0]
countries_h3.loc[countries_h3['name']=='Norway','geometry'] = norway_borders['geometry'].iloc[0]

In [ ]:
sample_counts = data_df[
    (data_df['dataset_name'].isin(processed_datasets)) &
    (~data_df['sampling_crop_name'].isin(not_crop_classes))
    ]['h3_l3_cell'].value_counts()
phase2_sample_counts = phase2_data_df[
    (phase2_data_df['dataset_name'].isin(processed_datasets)) &
    (~phase2_data_df['sampling_crop_name'].isin(not_crop_classes))
    ]['h3_l3_cell'].value_counts()
countries_h3['phase1_n_samples'] = countries_h3.index.map(sample_counts)
countries_h3['phase2_n_samples'] = countries_h3.index.map(phase2_sample_counts)

In [ ]:
# ['Africa','North America','Europe','South America','Asia']
continent = 'North America'

if continent=='Europe':
    continent_df = countries[
        (countries['continent']==continent) & 
        (countries['name']!='Russia')
        ]
    continent_h3_df = countries_h3[
        (countries_h3['continent']==continent) & 
        (countries_h3['name']!='Russia')
        ]
else:
    continent_df = countries[countries['continent']==continent]
    continent_h3_df = countries_h3[countries_h3['continent']==continent]

fig, axs = plt.subplots(
    ncols=2, 
    nrows=1, 
    figsize=(14,8),
    )
fig.suptitle("""
WorldCereal Phase I and II Samples Distribution, Cropland
{}
""".format(continent))
continent_df.plot(
    color='lightgrey',
    ax=axs.flatten()[0])
continent_h3_df.plot(
    column='phase1_n_samples', 
    cmap='Greens',
    ax=axs.flatten()[0],
    )
axs.flatten()[0].axis('off')
axs.flatten()[0].title.set_text("""
Phase I, Total samples: {}
""".format(int(continent_h3_df['phase1_n_samples'].sum())))
continent_df.plot(
    color='lightgrey',
    ax=axs.flatten()[1])
continent_h3_df.plot(
    column='phase2_n_samples', 
    cmap='Greens',
    ax=axs.flatten()[1],
    )
axs.flatten()[1].axis('off')
axs.flatten()[1].title.set_text("""
Phase II, Total samples: {}
""".format(int(continent_h3_df['phase2_n_samples'].sum())))

plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/djf_sampling_comparison_{}.png'.format(''.join(continent)), dpi=300)

In [ ]:
fig, axs = plt.subplots(
    ncols=2, 
    nrows=3, 
    figsize=(8,16),
    )
fig.tight_layout()
gs = axs[2, 1].get_gridspec()
for ax in axs[-1, -2:]:
    ax.remove()
axbig = fig.add_subplot(gs[-1,-2:])
fig.suptitle('WorldCereal Phase I Samples Distribution, Cropland', y=1.03)
for ii,continent in enumerate(['Africa','North America','Europe','South America','Asia']):
    if continent=='Europe':
        continent_df = countries[
            (countries['continent']==continent) & 
            (countries['name']!='Russia')
            ]
        continent_h3_df = countries_h3[
            (countries_h3['continent']==continent) & 
            (countries_h3['name']!='Russia')
            ]
        continent_df.plot(
            color='lightgrey',
            ax=axs.flatten()[ii])
        continent_h3_df.plot(
            column='n_samples', 
            cmap='Greens',
            ax=axs.flatten()[ii]
            )
        axs.flatten()[ii].xaxis.set_ticklabels([])
        axs.flatten()[ii].xaxis.set_ticks([])
        axs.flatten()[ii].yaxis.set_ticklabels([])
        axs.flatten()[ii].yaxis.set_ticks([])
        axs.flatten()[ii].title.set_text("""
        {}
        Total samples: {}
        """.format(continent, int(continent_h3_df['n_samples'].sum())))
    elif continent=='Asia':
        continent_df = countries[countries['continent']==continent]
        continent_h3_df = countries_h3[countries_h3['continent']==continent]
        continent_df.plot(
            color='lightgrey',
            ax=axbig)
        continent_h3_df.plot(
            column='n_samples', 
            cmap='Greens',
            ax=axbig)
        axbig.xaxis.set_ticklabels([])
        axbig.xaxis.set_ticks([])
        axbig.yaxis.set_ticklabels([])
        axbig.yaxis.set_ticks([])
        axbig.title.set_text("""
        {}
        Total samples: {}
        """.format(continent, int(continent_h3_df['n_samples'].sum())))
    elif continent=='Russia':
        continent_df = countries[(countries['name']=='Russia')]
        continent_h3_df = countries_h3[(countries_h3['name']=='Russia')]
    else:
        continent_df = countries[countries['continent']==continent]
        continent_h3_df = countries_h3[countries_h3['continent']==continent]
        continent_df.plot(
            color='lightgrey',
            ax=axs.flatten()[ii])
        continent_h3_df.plot(
            column='n_samples', 
            cmap='Greens',
            ax=axs.flatten()[ii],
            # legend=True
            )
        axs.flatten()[ii].xaxis.set_ticklabels([])
        axs.flatten()[ii].xaxis.set_ticks([])
        axs.flatten()[ii].yaxis.set_ticklabels([])
        axs.flatten()[ii].yaxis.set_ticks([])
        axs.flatten()[ii].title.set_text("""
        {}
        Total samples: {}
        """.format(continent, int(continent_h3_df['n_samples'].sum())))
plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/djf_sampling_phase1_cropland.png', dpi=300)

### Presto Embeddings

In [ ]:
# add pre-computed presto features 
# presto_df = pd.read_parquet('{}pretrainedprestofts-{}_calval.parquet'.format(tpath,freq))
presto_df = pd.read_parquet('{}prestofts-{}_calval.parquet'.format(tpath,freq))

presto_df.set_index('sample_id', inplace=True)
presto_emb_feats = [xx for xx in presto_df.columns if 'presto_ft' in xx]

data_df = pd.concat([data_df,presto_df[presto_emb_feats]], join='inner', axis=1)
del presto_df
gc.collect()

### SatCLIP embeddings

1. Get the embeddings themselves for train samples

In [ ]:
# again, here goes either path to a model, or git submodule can be created 🤷‍♀️
model_path = '/home/cbutsko/Desktop/cbutsko_experiments/satclip-resnet18-l10.ckpt'
model = get_satclip(model_path, device='cpu')

In [ ]:
satclip_df = prepare_satclip_embeddings(model, data_df)
satclip_emb_feats = [xx for xx in satclip_df.columns if 'satclip_ft' in xx]
data_df = pd.concat([data_df,satclip_df[satclip_emb_feats]], join='inner', axis=1)

del satclip_df
gc.collect()

2. Fit PCA on (part of) data, transform SatCLIP embeddings into PCs and append them to main dataframes

In [ ]:
n_rows = data_df.shape[0]
n_pcs = 32
satclip_pca_feats = ['satclip_PC{}'.format(jj) for jj in range(n_pcs)]

satclip_pca_model = PCA(n_components=n_pcs)
scaler = StandardScaler()
scaler.fit(data_df[satclip_emb_feats].sample(n=int(0.7*n_rows)).values)
satclip_emb_norm = scaler.transform(data_df[satclip_emb_feats].values)
satclip_pca_model.fit(satclip_emb_norm[np.random.randint(n_rows, size=int(0.7*n_rows)),:])
gc.collect()
satclip_pca = satclip_pca_model.transform(satclip_emb_norm)
satclip_pca_df = pd.DataFrame(
    satclip_pca, 
    columns=satclip_pca_feats, 
    index=data_df.index)

data_df = pd.concat([data_df,satclip_pca_df], join='inner', axis=1)

del satclip_pca_df
gc.collect()

### Define Features

In [ ]:
# group feature names for easier use
n_ts = 36
# n_ts = 12

optical12_feats = [xx for xx in data_df.columns if re.search(r'OPTICAL.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
sar12_feats = [xx for xx in data_df.columns if re.search(r'SAR.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
temp12_feats  = [xx for xx in data_df.columns if re.search(r'METEO-temp.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
prcp12_feats  = [xx for xx in data_df.columns if re.search(r'METEO-precip.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
dem_feats = ['DEM-alt-20m', 'DEM-slo-20m']
latlon_feats = ['lat','lon']
biome_feats = [xx for xx in data_df.columns if 'biome' in xx]

satclip_emb_feats = [xx for xx in data_df.columns if 'satclip_ft' in xx]
presto_emb_feats = [xx for xx in data_df.columns if 'presto_ft' in xx]

satclip_pca_feats = [xx for xx in data_df.columns if 'satclip_PC' in xx]

presto_input_feats = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats
phaseI_expert_feats = list(data_df.columns[:133]) + ['lat','lon']

In [ ]:
tpath = '/vitodata/worldcereal/features/benchmarking_features/calval/'
monthly_data_df = process_raw_features_input_df(
    '{}rawts-monthly_calval.parquet'.format(tpath),
    wc2ec_map,
    ec_map,
    label_columns,
    add_countries=True
    )

optical12_feats = [xx for xx in monthly_data_df.columns if re.search(r'OPTICAL.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]
data_df = pd.concat([data_df,monthly_data_df[optical12_feats]], join='inner', axis=1)
del monthly_data_df
gc.collect()

### Compute NDVI Peaks

In [ ]:
n_ts = 12
for ts in range(n_ts):
    B04 = data_df['OPTICAL-B04-ts{}-10m'.format(ts)]
    B08 = data_df['OPTICAL-B08-ts{}-10m'.format(ts)]
    _ndvi = (B08 - B04) / (B08 + B04)
    _ndvi[_ndvi.abs()>1] = np.nan
    _ndvi[_ndvi.abs()==0] = np.nan
    data_df['NDVI-ts{}-10m'.format(ts)] = _ndvi
    del B04,B08,_ndvi
    gc.collect()

ndvi_feats = [xx for xx in data_df.columns if re.search(r'NDVI.*ts({})-'.format('|'.join(map(str, list(range(n_ts))))), xx)]

In [ ]:
data_df['n_peaks'] = data_df[ndvi_feats].apply(lambda xx: find_peaks(xx, prominence=0.25, distance=3)[0].shape[0], axis=1)
data_df['valid_date_ind'] = ((data_df['valid_date'] - data_df['start_date']).dt.days/30).astype(int)
data_df['valid_date_doy'] = data_df['valid_date'].dt.dayofyear

In [ ]:
data_df['valid_peak_ind'] = np.nan
seasons_mask = pd.DataFrame(False, index=data_df.index, columns=data_df.columns)
pbar = tqdm(total=len(data_df[data_df['n_peaks']>0]))
for sample_id in data_df[data_df['n_peaks']>0].index:
    pbar.update(1)
    tobs = data_df.loc[sample_id]
    peaks = find_peaks(tobs[ndvi_feats], prominence=0.25, distance=3)
    valid_peak_ind = np.argmin(np.abs(peaks[0] - tobs['valid_date_ind']))
    
    data_df.at[sample_id,'valid_peak_ind'] = peaks[0][valid_peak_ind]

    season_start_ts = peaks[1]['left_bases'][valid_peak_ind]
    season_end_ts = peaks[1]['right_bases'][valid_peak_ind]
    cols_to_null_p1 = [xx for xx in data_df.columns if any(ts in xx for ts in ['-ts{}-'.format(ii) for ii in range(season_start_ts)])]
    cols_to_null_p2 = [xx for xx in data_df.columns if any(ts in xx for ts in ['-ts{}-'.format(ii) for ii in range(season_end_ts,12)])]
    cols_to_null = cols_to_null_p1 + cols_to_null_p2
    # seasons_mask.loc[sample_id][cols_to_null] = True

data_df['valid_peak_ind'].fillna(data_df['valid_date_ind'], inplace=True)

### Create test splits
0. Doubtful samples

In [ ]:
bad_eos_coutries = ['BFA','MEX','MLI','KAZ','NGA','UZB','SDN','SEN','DMA']

# samples2remove_fpath = 'resources/samples2remove.csv'
samples2remove_samples = data_df[data_df['country_code'].isin(bad_eos_coutries)].reset_index()['sample_id']
# samples2remove_samples.to_csv(samples2remove_fpath)

1a. Sanity check set: CROPTYPE

In [ ]:
sanity_check_test_split_fpath = '/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/croptype_sanity_check_test_split_samples.csv'

if os.path.isfile(sanity_check_test_split_fpath):
    sanity_check_test_split_samples = pd.read_csv(sanity_check_test_split_fpath)
else:
    single_labels = data_df[['worldcereal_croptype','country_code']].value_counts()
    single_labels = single_labels[single_labels==1]
    data_df['is_single_label'] = data_df.set_index(['worldcereal_croptype','country_code']).index.map(single_labels)

    _, tst_df = train_test_split(
        data_df[data_df['is_single_label'].isna()],
        stratify=data_df[data_df['is_single_label'].isna()][['worldcereal_croptype','country_code']],
        test_size=0.1,
        random_state=42)

    sanity_check_test_split_samples = tst_df.reset_index()['sample_id']
    sanity_check_test_split_samples.to_csv(sanity_check_test_split_fpath)

1b. Sanity check set: CROPLAND

In [ ]:
cropland_test_split_fpath = '/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/cropland_sanity_check_test_split_samples.csv'

if os.path.isfile(cropland_test_split_fpath):
    cropland_test_split_samples = pd.read_csv(cropland_test_split_fpath)
else:
    single_labels = data_df[['cropland_wc','country_code']].value_counts()
    single_labels = single_labels[single_labels==1]
    data_df['is_single_label'] = data_df.set_index(['cropland_wc','country_code']).index.map(single_labels)

    _, tst_df = train_test_split(
        data_df[data_df['is_single_label'].isna()],
        stratify=data_df[data_df['is_single_label'].isna()][['cropland_wc','country_code']],
        test_size=0.3,
        random_state=42)

    cropland_test_split_samples = tst_df.reset_index()['sample_id']
    cropland_test_split_samples.to_csv(cropland_test_split_fpath)

2a. Spatial generalization set: CROPLAND

In [ ]:
cropland_difficult_countries = [
    'ARG','TZA','ETH','LVA','ESP','NGA'
    ]
cropland_spatial_generalization_test_split_fpath = '/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/cropland_spatial_generalization_test_split_samples.csv'

if os.path.isfile(cropland_spatial_generalization_test_split_fpath):
    cropland_spatial_generalization_test_split_samples = pd.read_csv(cropland_spatial_generalization_test_split_fpath)
else:
    cropland_spatial_generalization_test_split_samples = data_df[data_df['country_code'].isin(cropland_difficult_countries)].reset_index()['sample_id']
    cropland_spatial_generalization_test_split_samples.to_csv(cropland_spatial_generalization_test_split_fpath)

2b. Spatial generalization set: CROPTYPE

In [ ]:
croptype_difficult_countries = [
    'FIN','GRC','IDN','ITA','MAR','MDG','MOZ','PRT','SOM',
    'ETH','ESP','AUT','BRA','TZA'
    ]
croptype_spatial_generalization_test_split_fpath = '/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/croptype_spatial_generalization_test_split_samples.csv'

if os.path.isfile(croptype_spatial_generalization_test_split_fpath):
    croptype_spatial_generalization_test_split_samples = pd.read_csv(croptype_spatial_generalization_test_split_fpath)
else:
    croptype_spatial_generalization_test_split_samples = data_df[data_df['country_code'].isin(croptype_difficult_countries)].reset_index()['sample_id']
    croptype_spatial_generalization_test_split_samples.to_csv(croptype_spatial_generalization_test_split_fpath)

3. Seasonality capturing set

In [ ]:
seasonality_test_countries = ['EGY','BRA','ETH','ITA','RWA']
seasonality_test_split_fpath = '/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/croptype_seasonality_test_split_samples.csv'

if os.path.isfile(seasonality_test_split_fpath):
    seasonality_test_split_samples = pd.read_csv(seasonality_test_split_fpath)
else:
    seasonality_test_split_samples = data_df[
        (data_df['country_code'].isin(seasonality_test_countries)) & 
        (data_df['n_peaks']>1)].reset_index()['sample_id']
    seasonality_test_split_samples.to_csv(seasonality_test_split_fpath)

3. Time generalization set

In [ ]:
test_years = [2021]
time_generalization_test_split_fpath = '/home/cbutsko/Desktop/worldcereal-classification/notebooks/resources/time_generalization_test_split_samples.csv'

if os.path.isfile(time_generalization_test_split_fpath):
    time_generalization_test_split_samples = pd.read_csv(time_generalization_test_split_fpath)
else:
    time_generalization_test_split_samples = data_df[
    (data_df['year'].isin(test_years)) & 
    (~data_df.index.isin(samples2remove_samples))
    ].reset_index()['sample_id']
    time_generalization_test_split_samples.to_csv(time_generalization_test_split_fpath)

### Plot NDVI timeseries fo country/crop/n_peaks

In [ ]:
# sample_size = 100
# n_peaks = 2
# country = 'ETH'
# crop = 'Maize'
    
# _ndvi_subset = data_df[
#     (data_df['CROPTYPE_NAME']==crop) & 
#     (data_df['country_code']==country) &
#     (data_df['n_peaks']==n_peaks)
#     # (data_df['crop_prob']>0.7)
#     ]
# n_points = len(_ndvi_subset)
# if n_points>sample_size:
#     ndvis = _ndvi_subset.sample(n=sample_size).transpose()
# else:
#     ndvis = _ndvi_subset.transpose()
#     sample_size = n_points
    
# # ndvis.loc[ndvi_feats].fillna(method='ffill', axis=0, inplace=True)

# ndvis.loc[ndvi_feats].plot(figsize=(15,9), legend=None)
# for ii in range(ndvis.shape[-1]): 
#     plt.axvline(ndvis.loc['valid_date_ind'].iloc[ii], linewidth=0.8) 
# # plt.title(f"""
# #     {crop}, {country} 
# #     Random {sample_size} samples
# #     (vertical lines are valid_date)
# # """)
# # plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}.png'.format(crop,country), dpi=300)
# plt.title(f"""
#     {crop}, {country} 
#     Random {sample_size} samples with {n_peaks} detected peaks
#     (vertical lines are valid_date)
# """)
# plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_peaks={}.png'.format(crop,country,n_peaks), dpi=300)

In [ ]:
# sample_size = 100
# n_peaks = 3
# country = 'MOZ'
# crop = 'Winter wheat'

# for predicted_as in ['crop','not_crop']:
    
#     if predicted_as=='crop':
#         _ndvi_subset = data_df[
#             (data_df['CROPTYPE_NAME']==crop) & 
#             (data_df['country_code']==country) &
#             # (data_df['n_peaks']==n_peaks)
#             (data_df['crop_prob']>0.7)
#             ]
#     else:
#         _ndvi_subset = data_df[
#             (data_df['CROPTYPE_NAME']==crop) & 
#             (data_df['country_code']==country) &
#             (data_df['crop_prob']<0.3)
#             ]

#     n_points = len(_ndvi_subset)
#     if n_points>sample_size:
#         ndvis = _ndvi_subset.sample(n=sample_size).transpose()
#     else:
#         ndvis = _ndvi_subset.transpose()
#         sample_size = n_points
        
#     # ndvis.loc[ndvi_feats].fillna(method='ffill', axis=0, inplace=True)

#     ndvis.loc[ndvi_feats].plot(figsize=(15,9), legend=None)
#     for ii in range(ndvis.shape[-1]): 
#         plt.axvline(ndvis.loc['valid_date_ind'].iloc[ii], linewidth=0.8) 
#     plt.title(f"""
#         {crop}, {country} 
#         Random {sample_size} samples confidently predicted as {predicted_as}
#         (vertical lines are valid_date)
#     """)
#     plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_predicted_{}.png'.format(crop,country,predicted_as), dpi=300)
#     # plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_predicted_crop.png'.format(crop,country), dpi=300)

#     # plt.title(f"""
#     #     {crop}, {country} 
#     #     Random {sample_size} samples with {n_peaks} detected peaks
#     #     (vertical lines are valid_date)
#     # """)
#     # plt.savefig('/home/cbutsko/Desktop/cbutsko_experiments/crop_charts/{}_{}_peaks={}.png'.format(crop,country,n_peaks), dpi=300)

In [ ]:
# attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks','valid_date','start_date','end_date','crop_prob']
# data_df[data_df['crop_prob']<0.3][attr_lst].to_csv('/home/cbutsko/Desktop/cbutsko_experiments/suspicious_cropland_predicted_as_not_crop.csv')

### Test _find_peaks_ algo and plot NDVI and peaks found

In [ ]:
colors = list(dict(mcolors.BASE_COLORS, **mcolors.CSS4_COLORS).keys())
random.shuffle(colors)

country = 'ESP'
n_peaks = 2

_ndvi_subset = data_df[
    (data_df['country_code']==country) &
    (data_df['CROPTYPE_NAME']==crop) &
    (data_df['n_peaks']==n_peaks)
    ]
    
n_points = len(_ndvi_subset)
if n_points>sample_size:
    ndvis = _ndvi_subset.sample(n=sample_size)[ndvi_feats].transpose()
else:
    ndvis = _ndvi_subset[ndvi_feats].transpose()

In [ ]:
tind = np.random.choice(range(sample_size))

tndvi = ndvis.iloc[:,tind]
peaks = find_peaks(tndvi, prominence=0.25, distance=3)
tndvi.plot(figsize=(12,5))
if peaks[0].shape[0]>0:
    for ii in range(peaks[0].shape[0]):
        plt.axvline(peaks[0][ii], c=colors[ii], linewidth=1.2)
        plt.axvline(peaks[1]['left_bases'][ii], c=colors[ii], linewidth=0.8, linestyle='dashed')
        plt.axvline(peaks[1]['right_bases'][ii], c=colors[ii], linewidth=0.8, linestyle='dashed')

## Classification

### Binary crop/not_crop

In [ ]:
year_loo_samples = data_df[
    (data_df['year']==2021) & 
    (~data_df.index.isin(samples2remove_samples))
    ].reset_index()['sample_id']

### Realms Training

In [ ]:
label_col = 'cropland_wc'

# split_type = 'random-split'
# tst_inds = cropland_test_split_samples['sample_id']

split_type = 'spatial-split'
tst_inds = cropland_spatial_generalization_test_split_samples['sample_id']

# split_type = 'bad-eos-samples'
# tst_inds = samples2remove_samples

# split_type = 'year-loo-samples'
# tst_inds = year_loo_samples

# features_set = 'phaseI-expert-feats'
# features_list = phaseI_expert_feats

features_set = 'presto-input'
features_list = presto_input_feats

# features_set = 'presto-pretrained'
# features_list = presto_emb_feats

model_params = {
    'depth':8,
    'learning_rate':0.3,
    'l2_leaf_reg':100
    }

preds_df = pd.DataFrame()

for realm_id in range(2,8):
    print('Training model for {}...'.format(realm_names[realm_id]))
    realm_data = data_df[data_df['REALM']==realm_id]

    # realm_data = realm_data.sample(frac=1)
    # n_splits = 3
    # split_inds = np.array_split(range(len(realm_data)), n_splits)
    # for tsplit in range(n_splits):
    #     trn_inds = [x for xs in [xx for ii,xx in enumerate(split_inds) if ii!=tsplit]  for x in xs]
    #     val_inds = split_inds[tsplit]

    #     trnval_df = realm_data.iloc[trn_inds]
    #     tst_df = realm_data.iloc[val_inds]

    trnval_df = realm_data[~realm_data.index.isin(tst_inds)]
    tst_df = realm_data[realm_data.index.isin(tst_inds)]

    trn_df, val_df = train_test_split(
        trnval_df,
        stratify=trnval_df[label_col],
        test_size=0.3,
        random_state=42)

    X_trn_df = trn_df[features_list]
    y_trn_df = trn_df[label_col]
    X_val_df = val_df[features_list]
    y_val_df = val_df[label_col]
    X_tst_df = tst_df[features_list]
    y_tst_df = tst_df[label_col]

    model = CatBoostClassifier(
    iterations=2000, 
    depth=model_params['depth'],
    eval_metric='F1',
    learning_rate=model_params['learning_rate'],
    l2_leaf_reg=model_params['l2_leaf_reg'],
    verbose=250,
    random_seed=42,
    )

    model.fit(
        Pool(X_trn_df,y_trn_df), 
        eval_set=Pool(X_val_df,y_val_df), 
        early_stopping_rounds=200)
    pred = model.predict(X_tst_df).flatten()

    print('F1 for {}: {:2f}'.format(realm_names[realm_id], f1_score(y_tst_df.values, pred)))

    realm_preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    realm_preds_df.columns = ['true','pred']
    realm_preds_df.set_index(tst_df.index, inplace=True)
    realm_preds_df['country_code'] = tst_df['country_code']

    preds_df = pd.concat((preds_df,realm_preds_df), axis=0)

    gc.collect()

In [ ]:
f1_score(preds_df['true'], preds_df['pred'])

In [ ]:
preds_df.groupby(['country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['true'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()

In [ ]:
preds_df[preds_df.index.isin(cropland_spatial_generalization_test_split_samples['sample_id'])].groupby(['country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['true'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()

In [ ]:
f1_score(
    preds_df[preds_df.index.isin(cropland_spatial_generalization_test_split_samples['sample_id'])]['true'], 
    preds_df[preds_df.index.isin(cropland_spatial_generalization_test_split_samples['sample_id'])]['pred']
    )

In [ ]:
f1_score(
    preds_df[preds_df.index.isin(year_loo_samples)]['true'], 
    preds_df[preds_df.index.isin(year_loo_samples)]['pred']
    )

### Single Model Training

In [ ]:
label_col = 'cropland_wc'

# split_type = 'random-split'
# tst_inds = cropland_test_split_samples['sample_id']

split_type = 'spatial-split'
tst_inds = cropland_spatial_generalization_test_split_samples['sample_id']

# split_type = 'bad-eos-samples'
# tst_inds = samples2remove_samples

# split_type = 'year-loo-samples'
# tst_inds = year_loo_samples

# features_set = 'phaseI-expert-feats'
# features_list = phaseI_expert_feats

# features_set = 'presto-input'
# features_list = presto_input_feats

features_set = 'presto-finetuned-cropland'
features_list = presto_emb_feats

# features_set = 'presto-pretrained'
# features_list = presto_emb_feats

trnval_df = data_df[~data_df.index.isin(tst_inds)]
tst_df = data_df[data_df.index.isin(tst_inds)]

trn_df, val_df = train_test_split(
    trnval_df,
    stratify=trnval_df[label_col],
    test_size=0.3,
    random_state=42)

X_trn_df = trn_df[features_list]
y_trn_df = trn_df[label_col]
X_val_df = val_df[features_list]
y_val_df = val_df[label_col]
X_tst_df = tst_df[features_list]
y_tst_df = tst_df[label_col]

In [ ]:
model_params = {
    'depth':8,
    'learning_rate':0.3,
    'l2_leaf_reg':100
    }

In [ ]:
model = CatBoostClassifier(
    iterations=2000, 
    depth=model_params['depth'],
    eval_metric='F1',
    learning_rate=model_params['learning_rate'],
    l2_leaf_reg=model_params['l2_leaf_reg'],
    verbose=100,
    random_seed=42,
    )

model.fit(
    Pool(X_trn_df,y_trn_df), 
    eval_set=Pool(X_val_df,y_val_df), 
    early_stopping_rounds=200)
pred = model.predict(X_tst_df).flatten()

# # save model
# model_path = 'resources/binary_cropland_model_{}_{}_depth={}_lr={}_l2leafreg={}.cbm'.format(
#     features_set,
#     split_type,
#     model_params['depth'],
#     model_params['learning_rate'],
#     model_params['l2_leaf_reg'],
#     )
# model.save_model(model_path)

In [ ]:
f1_score(y_tst_df.values, pred)

In [ ]:
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df.set_index(tst_df.index, inplace=True)
preds_df['country_code'] = tst_df['country_code']

In [ ]:
preds_df.groupby(['country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['true'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()

In [ ]:
data_df = data_df.sample(frac=1)
n_splits = 3
split_inds = np.array_split(range(len(data_df)), n_splits)

preds_df = pd.DataFrame()
for tsplit in range(n_splits):
    trn_inds = [x for xs in [xx for ii,xx in enumerate(split_inds) if ii!=tsplit]  for x in xs]
    val_inds = split_inds[tsplit]

    trnval_df = data_df.iloc[trn_inds]
    tst_df = data_df.iloc[val_inds]

    trn_df, val_df = train_test_split(
        trnval_df,
        stratify=trnval_df[label_col],
        test_size=0.3,
        random_state=42)

    X_trn_df = trn_df[features_list]
    y_trn_df = trn_df[label_col]
    X_val_df = val_df[features_list]
    y_val_df = val_df[label_col]
    X_tst_df = tst_df[features_list]
    y_tst_df = tst_df[label_col]

    model = CatBoostClassifier(
        iterations=2000, 
        depth=model_params['depth'],
        eval_metric='F1',
        learning_rate=model_params['learning_rate'],
        l2_leaf_reg=model_params['l2_leaf_reg'],
        verbose=250,
        random_seed=42,
        )

    model.fit(
        Pool(X_trn_df,y_trn_df), 
        eval_set=Pool(X_val_df,y_val_df), 
        early_stopping_rounds=200)
    pred = model.predict(X_tst_df).flatten()

    _preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    _preds_df.columns = ['true','pred']
    _preds_df.set_index(tst_df.index, inplace=True)
    _preds_df['country_code'] = tst_df['country_code']
    preds_df = pd.concat((preds_df,_preds_df), axis=0)

    gc.collect()

In [ ]:
f1_score(
    preds_df[preds_df.index.isin(cropland_spatial_generalization_test_split_samples['sample_id'])]['true'], 
    preds_df[preds_df.index.isin(cropland_spatial_generalization_test_split_samples['sample_id'])]['pred']
    )

In [ ]:
preds_df[preds_df.index.isin(cropland_spatial_generalization_test_split_samples['sample_id'])].groupby(['country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['true'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()

In [ ]:
f1_score(
    preds_df[preds_df.index.isin(year_loo_samples)]['true'], 
    preds_df[preds_df.index.isin(year_loo_samples)]['pred']
    )

In [ ]:
f1_score(
    preds_df[preds_df.index.isin(samples2remove_samples)]['true'], 
    preds_df[preds_df.index.isin(samples2remove_samples)]['pred']
    )

### Multiclass Vanilla Classification

In [ ]:
data_df = data_df[data_df['LANDCOVER_LABEL']==11]
classes_to_exclude = [
    991, 7900, 9900, 9998,
    1910,1900,1920,1000,
    11,9910,6212
]
data_df = data_df[~data_df['CROPTYPE_LABEL'].isin(classes_to_exclude)]
gc.collect()

In [ ]:
# label_col = 'worldcereal_croptype'
label_col = 'worldcereal_croptype_small'
attr_lst = ['ref_id','lat','lon','country_code']
class_map = code2name_map
# class_map = wc_modest_classes

# split_type = 'random-split'
# tst_inds = sanity_check_test_split_samples['sample_id']

# split_type = 'spatial-split'
tst_inds = croptype_spatial_generalization_test_split_samples['sample_id']

# split_type = 'year-loo-samples'
# tst_inds = year_loo_samples

# tst_inds = seasonality_test_split_samples['sample_id']

# features_list = phaseI_expert_feats
features_list = presto_input_feats
# features_list = presto_emb_feats

# features_list = features_list + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks'] 
# + ndvi_feats

In [ ]:
trnval_df = data_df[~data_df.index.isin(tst_inds)]
tst_df = data_df[data_df.index.isin(tst_inds)]
# trn_df = trn_df[trn_df['crop_prob']>0.7]

trn_df, val_df = train_test_split(
    trnval_df,
    stratify=trnval_df[label_col],
    test_size=0.3,
    random_state=42)

X_trn_df = trn_df[features_list]
y_trn_df = trn_df[label_col].map(class_map)
X_val_df = val_df[features_list]
y_val_df = val_df[label_col].map(class_map)
X_tst_df = tst_df[features_list]
y_tst_df = tst_df[label_col].map(class_map)

# initialize and train the model
model = CatBoostClassifier(
    iterations=500, 
    depth=8,
    eval_metric='TotalF1',
    learning_rate=0.6,
    l2_leaf_reg=30,
    verbose=25,
    random_seed=42,
    )

model.fit(
    Pool(X_trn_df,y_trn_df), 
    eval_set=Pool(X_val_df,y_val_df), 
    early_stopping_rounds=50
    )

pred = pd.Series(model.predict(X_tst_df).flatten())
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df.set_index(tst_df.index, inplace=True)
preds_df[attr_lst] = tst_df[attr_lst]
preds_df.reset_index(inplace=True)
gc.collect()

In [ ]:
# preds_df.loc[preds_df['true'].isin(other_classes),'true'] = 'other'
# preds_df.loc[preds_df['pred'].isin(other_classes),'pred'] = 'other'

In [ ]:
print(classification_report(
    preds_df['true'], 
    preds_df['pred']
    ))

In [ ]:
preds_df_vanilla = preds_df.copy()

In [ ]:
preds_df['country_code'].value_counts()

In [ ]:
print(classification_report(
    preds_df[preds_df['country_code']=='AUT']['true'], 
    preds_df[preds_df['country_code']=='AUT']['pred']
    ))

In [ ]:
cm = confusion_matrix(preds_df_vanilla['true'], preds_df_vanilla['pred'])
plot_confusion_matrix(cm, np.unique(preds_df_vanilla['true']))

### Hierachical Classification

In [ ]:
# label_columns.remove('cropland')
# label_columns.remove('cropgroup')

In [ ]:
data_df.fillna(0, inplace=True)

In [ ]:
label_columns_bckp = ['cropland', 'landcover', 'cropgroup', 'croptype', 'worldcereal_croptype']
other_classes = [xx for xx in data_df['worldcereal_croptype'].map(code2name_map).unique() if xx not in wc_modest_classes.values()]

In [ ]:
label_columns = ['landcover', 'worldcereal_croptype']
# label_columns = ['landcover', 'worldcereal_croptype_small']
attr_lst = ['ref_id','lat','lon','country_code']
tst_inds = croptype_spatial_generalization_test_split_samples['sample_id']
# tst_inds = sanity_check_test_split_samples['sample_id']
class_map = code2name_map
# class_map = wc_modest_classes

features_list = presto_emb_feats
# features_list = phaseI_expert_feats
features_list = features_list + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks'] 

In [ ]:
trnval_df = data_df[~data_df.index.isin(tst_inds)]
tst_df = data_df[data_df.index.isin(tst_inds)]

X_trn_df = trnval_df[features_list]
y_trn_df = trnval_df[label_columns].apply(lambda xx: xx.map(class_map), axis=0)
X_tst_df = tst_df[features_list]
y_tst_df = tst_df[label_columns].apply(lambda xx: xx.map(class_map), axis=0)

In [ ]:
cb_clsfr = CatBoostClassifierWrapper(
    iterations=2000, 
    depth=8,
    eval_metric='F1',
    learning_rate=0.3,
    l2_leaf_reg=100,
    verbose=50,
    random_seed=42,
)

classifier = LocalClassifierPerNode(
    local_classifier=cb_clsfr,
    n_jobs=-1,
    # binary_policy='inclusive',
)

classifier.fit(X_trn_df, y_trn_df)
pred = classifier.predict(X_tst_df)

level_true = y_tst_df[label_columns[-1]].values
level_pred = pred[:,-1]

preds_df = pd.DataFrame([level_true, level_pred]).transpose()
preds_df.columns = ['true','pred']
preds_df.set_index(tst_df.index, inplace=True)
preds_df[attr_lst] = tst_df[attr_lst]
preds_df.reset_index(inplace=True)
# preds_df.loc[preds_df['true'].isin(other_classes),'true'] = 'other'
# preds_df.loc[preds_df['pred'].isin(other_classes),'pred'] = 'other'
gc.collect()

In [ ]:
print(classification_report(
    preds_df['true'], 
    preds_df['pred']
    )) 

In [ ]:
cm = confusion_matrix(
    preds_df['true'], 
    preds_df['pred'])
plot_confusion_matrix(cm, np.unique(preds_df['true']))

In [ ]:
cm = confusion_matrix(
    preds_df_vanilla['true'], 
    preds_df_vanilla['pred'])
plot_confusion_matrix(cm, np.unique(preds_df_vanilla['true']))

In [ ]:
preds_df_vanilla['country_code'].value_counts()

In [ ]:
clsfc_report = pd.DataFrame(classification_report(
    preds_df[~preds_df['country_code'].isin(['AUT','ESP'])]['true'], 
    preds_df[~preds_df['country_code'].isin(['AUT','ESP'])]['pred'],
    output_dict=True
    )).transpose()
clsfc_report = clsfc_report[['f1-score','support']]
clsfc_report.columns = ['hiclass_f1','support']
_clsfc_report = pd.DataFrame(classification_report(
    preds_df_vanilla[~preds_df_vanilla['country_code'].isin(['AUT','ESP'])]['true'], 
    preds_df_vanilla[~preds_df_vanilla['country_code'].isin(['AUT','ESP'])]['pred'],
    output_dict=True
    )).transpose()
clsfc_report['vanilla_f1'] = _clsfc_report[['f1-score']]
clsfc_report = clsfc_report[['vanilla_f1','hiclass_f1','support']]

clsfc_report[clsfc_report['support']>0]


In [ ]:
tcountry = 'ESP'

clsfc_report = pd.DataFrame(classification_report(
    preds_df[preds_df['country_code']==tcountry]['true'], 
    preds_df[preds_df['country_code']==tcountry]['pred'],
    output_dict=True
    )).transpose()
clsfc_report = clsfc_report[['f1-score','support']]
clsfc_report.columns = ['hiclass_f1','support']
_clsfc_report = pd.DataFrame(classification_report(
    preds_df_vanilla[preds_df_vanilla['country_code']==tcountry]['true'], 
    preds_df_vanilla[preds_df_vanilla['country_code']==tcountry]['pred'],
    output_dict=True
    )).transpose()
clsfc_report['vanilla_f1'] = _clsfc_report[['f1-score']]
clsfc_report = clsfc_report[['vanilla_f1','hiclass_f1','support']]

clsfc_report[clsfc_report['support']>0]


In [ ]:
cm = confusion_matrix(
    preds_df[preds_df['country_code']==tcountry]['true'], 
    preds_df[preds_df['country_code']==tcountry]['pred'])
plot_confusion_matrix(cm, np.unique(preds_df[preds_df['country_code']==tcountry]['true']))

In [ ]:
cm = confusion_matrix(
    preds_df_vanilla[preds_df_vanilla['country_code']==tcountry]['true'], 
    preds_df_vanilla[preds_df_vanilla['country_code']==tcountry]['pred'])
plot_confusion_matrix(cm, np.unique(preds_df_vanilla[preds_df_vanilla['country_code']==tcountry]['true']))

In [ ]:
features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks']
# optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats
#  + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks']
# + ['valid_date_doy']
# + ['valid_date_ind']
# + ['valid_peak_ind']
# + ['n_peaks']
# presto_emb_feats

In [ ]:
trn_df, val_df = train_test_split(
    data_df,
    stratify=data_df['croptype'],
    test_size=0.3,
    random_state=42)

In [ ]:
test_country = 'EGY'
injection_size = 0.1

trn_df = data_df[data_df['country_code']!=test_country]
val_df = data_df[data_df['country_code']==test_country]

single_sample_classes = val_df['croptype'].value_counts()[val_df['croptype'].value_counts()==1].index
if len(single_sample_classes)>0:
    val_df = val_df[~val_df[label_col].isin(single_sample_classes)]

_, injection_df = train_test_split(
    val_df,
    stratify=val_df['croptype'],
    test_size=injection_size,
    random_state=42)
trn_df = pd.concat((trn_df,injection_df), axis=0)

In [ ]:
injection_df['croptype'].map(tmap).value_counts()

In [ ]:
# trn_df = trn_df[trn_df['crop_prob']>0.7]

X_trnval_df = trn_df[features_list]
y_trnval_df = trn_df[label_col].map(tmap)
X_tst_df = val_df[features_list]
y_tst_df = val_df[label_col].map(tmap)

# initialize and train the model
model = CatBoostClassifier(
    iterations=500, 
    depth=8,
    eval_metric='TotalF1',
    learning_rate=0.1,
    # l2_leaf_reg=150,
    verbose=50,
    random_seed=42,
    )

model.fit(X_trnval_df, y_trnval_df)
pred = pd.Series(model.predict(X_tst_df).flatten())

# create dataframe with predictions and useful attributes
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df.set_index(val_df.index, inplace=True)
attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks']
preds_df[attr_lst] = val_df[attr_lst]

In [ ]:
print(classification_report(
    preds_df[preds_df['n_peaks']==0]['true'], 
    preds_df[preds_df['n_peaks']==0]['pred']
    )) 

In [ ]:
print(classification_report(
    preds_df[preds_df['n_peaks']==1]['true'], 
    preds_df[preds_df['n_peaks']==1]['pred']
    )) 

In [ ]:
print(classification_report(
    preds_df[preds_df['n_peaks']>1]['true'], 
    preds_df[preds_df['n_peaks']>1]['pred']
    )) 

In [ ]:
cm = confusion_matrix(preds_df['true'], preds_df['pred'])
plot_confusion_matrix(cm, np.unique(list(np.unique(preds_df['true'].unique())) + list(np.unique(preds_df['pred'].unique()))))

In [ ]:
preds_df.groupby(['n_peaks']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()

In [ ]:
preds_df[preds_df['n_peaks']>1].groupby(['country_code']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index().sort_values(by='n_maize_pixels', ascending=False).iloc[:20]

In [ ]:
tres = preds_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()
tres.to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_4fold_cv_raw_ts.csv')

In [ ]:
data_df = data_df_bckp.copy()

data_df[seasons_mask] = np.nan
data_df.fillna(65535, inplace=True)

In [ ]:
features_for_aggregation = [
    'OPTICAL-B02','OPTICAL-B03','OPTICAL-B04','OPTICAL-B05',
    'OPTICAL-B06','OPTICAL-B07','OPTICAL-B08','OPTICAL-B8A',
    'OPTICAL-B11','OPTICAL-B12',
    'NDVI',
    'SAR-VV','SAR-VH',
    'METEO-temperature_mean','METEO-precipitation_flux'
    ]
tcols = [xx for xx in data_df.columns if any([kk in xx for kk in features_for_aggregation])]
data_df[data_df[tcols]>5000][tcols] = np.nan
data_df[data_df[tcols]==0][tcols] = np.nan

pbar = tqdm(total=len(features_for_aggregation)*1)
for tfeature in features_for_aggregation:
    tcols = [xx for xx in data_df.columns if tfeature in xx]
    new_cols = [
        '{}_agg_min'.format(tfeature),
        '{}_agg_mean'.format(tfeature),
        '{}_agg_max'.format(tfeature),
        '{}_agg_std'.format(tfeature),
        ]
    data_df[new_cols] = data_df[tcols].agg(['min','mean','max','std'], axis=1)
    pbar.update(1)

aggregated_features = [xx for xx in data_df.columns if '_agg_' in xx]

In [ ]:
# data_df_no_seasons_agg = data_df.copy()
data_df_seasons_agg = data_df.copy()

In [ ]:
# data_df = data_df_no_seasons_agg.copy()
# data_df = data_df_seasons_agg.copy()

# data_df = data_df_bckp.copy()
# data_df[seasons_mask] = np.nan
# data_df.fillna(65535, inplace=True)

In [ ]:
min_points_per_country = 50
# features_list = aggregated_features + ['valid_date_ind']
# ['valid_date_doy','n_peaks']
features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats + ['valid_date_ind']
# features_list = presto_emb_feats

countries_lst = data_df[data_df['CROPTYPE_NAME']==crop]['country_code'].value_counts()
countries_lst = countries_lst[countries_lst>min_points_per_country].index

preds_df = pd.DataFrame()
pbar = tqdm(total=len(countries_lst))
for test_country in countries_lst:
    pbar.update(1)

    trn_df = data_df[data_df['country_code']!=test_country]
    val_df = data_df[data_df['country_code']==test_country]

    # trn_df = trn_df[trn_df['crop_prob']>0.7]

    X_trnval_df = trn_df[features_list]
    y_trnval_df = trn_df[label_col]
    X_tst_df = val_df[features_list]
    y_tst_df = val_df[label_col]

    class_weights = {'False':1, 'True':1}

    model = CatBoostClassifier(
        iterations=500, 
        depth=8,
        eval_metric='F1',
        learning_rate=0.05,
        l2_leaf_reg=130,
        verbose=0,
        random_seed=42,
        class_weights=class_weights
        )

    model.fit(X_trnval_df, y_trnval_df)
    probs = model.predict_proba(X_tst_df)
    pred = model.predict(X_tst_df).flatten()
    if y_tst_df.nunique()==2:
        pred = np.array([xx=='True' for xx in pred])

    # create dataframe with predictions and useful attributes
    _preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    _preds_df.columns = ['true','pred']
    _preds_df['crop_prob'] = probs[:,1]
    _preds_df.set_index(val_df.index, inplace=True)
    attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks']
    _preds_df[attr_lst] = val_df[attr_lst]
    preds_df = pd.concat((preds_df,_preds_df), axis=0)

    gc.collect()

tres = preds_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
    'n_pixels': xx['ref_id'].count(),
    'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
    'f1': f1_score(xx['true'], xx['pred']),
    'precision': precision_score(xx['true'], xx['pred']),
    'recall': recall_score(xx['true'], xx['pred']),
    })).reset_index()
tres.to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_country_loo_raw_ts_valid_date_loc.csv')

In [ ]:
tres

In [ ]:
data_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
    'n_cropland_pixels': xx['ref_id'].count(),
    'n_confident_cropland_pixels': xx[xx['crop_prob']>0.7]['ref_id'].count(),
    })).reset_index().to_csv('/home/cbutsko/Desktop/cbutsko_experiments/maize_stats_confident_samples.csv')

In [ ]:
label_col

In [ ]:
tlist = [1,10,20,30,40,50,100,200,250,300,400,500,750]
# countries_lst = ['BEL', 'USA', 'AUT', 'FRA', 'ARG', 'ESP', 'UKR', 'CAN', 'NLD', 'DEU', 'NGA', 'RWA', 'LVA', 'POL', 'MOZ', 'UGA', 'BRA', 'ROU', 'ITA']
countries_lst = ['EGY','BRA','ETH','ITA','BEL','USA','CAN','RWA']

# res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_presto_emb_test_country_injection.csv'
res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_raw_ts_peak_encoding_test_country_injection.csv'
if os.path.isfile(res_fpath):
    maize_injection_res = pd.read_csv(res_fpath)
    if 'Unnamed: 0' in maize_injection_res.columns:
        maize_injection_res.drop('Unnamed: 0', axis=1, inplace=True)
    computed_countries = maize_injection_res['country'].unique()
    countries_lst = [xx for xx in countries_lst if xx not in computed_countries]
else: 
    maize_injection_res = pd.DataFrame()

pbar = tqdm(total=len(tlist)*len(countries_lst))
for test_country in countries_lst:
    for n_test_samples in tlist:
        pbar.update(1)
        trn_df = data_df[data_df['country_code']!=test_country]
        val_df = data_df[data_df['country_code']==test_country]

        if (val_df[label_col]).sum()<n_test_samples: continue

        test_injection_df = val_df[val_df[label_col]].sample(n=n_test_samples, random_state=42)
        trn_df = pd.concat((trn_df,test_injection_df), axis=0)

        trn_df = trn_df[trn_df['crop_prob']>0.7]

        # features_list = aggregated_features + ['valid_date_ind','n_peaks','valid_date_doy']
        features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks']
        # features_list = presto_emb_feats
        # features_list = ['valid_date_ind','n_peaks'] + optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats

        X_trnval_df = trn_df[features_list]
        y_trnval_df = trn_df[label_col]
        X_tst_df = val_df[features_list]
        y_tst_df = val_df[label_col]

        class_weights = {'False':1, 'True':1}

        model = CatBoostClassifier(
            iterations=500, 
            depth=8,
            eval_metric='F1',
            learning_rate=0.05,
            l2_leaf_reg=130,
            verbose=0,
            random_seed=42,
            class_weights=class_weights
            )

        model.fit(X_trnval_df, y_trnval_df)
        probs = model.predict_proba(X_tst_df)
        pred = model.predict(X_tst_df).flatten()
        if y_tst_df.nunique()==2:
            pred = np.array([xx=='True' for xx in pred])

        # create dataframe with predictions and useful attributes
        preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
        preds_df.columns = ['true','pred']
        # preds_df['crop_prob'] = probs[:,1]
        preds_df.set_index(val_df.index, inplace=True)
        # attr_lst = ['ref_id','lat','lon','CROPTYPE_NAME','country_code','n_peaks','crop_prob']
        # preds_df[attr_lst] = val_df[attr_lst]

        gc.collect()

        # tres = preds_df.groupby(['country_code','n_peaks']).apply(lambda xx: pd.Series({
        #     'n_pixels': xx['ref_id'].count(),
        #     'n_maize_pixels': xx[xx['CROPTYPE_NAME']=='Maize']['ref_id'].count(),
        #     'f1': f1_score(xx['true'], xx['pred']),
        #     'precision': precision_score(xx['true'], xx['pred']),
        #     'recall': recall_score(xx['true'], xx['pred']),
        #     })).reset_index()

        _f1 = f1_score(preds_df['true'], preds_df['pred'])
        tres = pd.DataFrame([test_country,n_test_samples,_f1]).transpose()
        tres.columns = ['country','n_test_samples','f1']
        maize_injection_res = pd.concat((maize_injection_res,tres), axis=0)
        maize_injection_res.to_csv(res_fpath)

In [ ]:
res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_presto_emb_test_country_injection.csv'
# res_fpath = '/home/cbutsko/Desktop/cbutsko_experiments/maize_raw_ts_test_country_injection.csv'
if os.path.isfile(res_fpath):
    maize_injection_res = pd.read_csv(res_fpath)
    if 'Unnamed: 0' in maize_injection_res.columns:
        maize_injection_res.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
maize_injection_res['n_test_samples'] = maize_injection_res['n_test_samples'].astype(int)
maize_injection_res['f1'] = maize_injection_res['f1'].astype(float)

In [ ]:
missing_piece = pd.DataFrame([
['POL',200,np.nan],
['MOZ',200,np.nan],
['UGA',200,np.nan],
['BRA',200,np.nan],
['ROU',200,np.nan],
['ITA',200,np.nan],
['POL',250,np.nan],
['MOZ',250,np.nan],
['UGA',250,np.nan],
['BRA',250,np.nan],
['ROU',250,np.nan],
['ITA',250,np.nan]
], columns=maize_injection_res.columns)
maize_injection_res = pd.concat((maize_injection_res,missing_piece), axis=0)
maize_injection_res.reset_index(drop=True, inplace=True)

In [ ]:
maize_injection_res['n_test_samples'] = maize_injection_res['n_test_samples'].astype(int)

In [ ]:
maize_injection_res = maize_injection_res.pivot(index='n_test_samples', columns='country', values='f1')
maize_injection_res[maize_injection_res==0] = np.nan

In [ ]:
maize_injection_res

In [ ]:
maize_injection_res.plot(figsize=(12,5))
plt.legend(loc='right')
plt.xlabel('n injected samples')
plt.title("""
Binary maize F1 score change depending on "injection" size from unseen country
Finetuned Presto embeddings
""")

In [ ]:
test_country = 'ARG'

trn_df = data_df.sample(frac=0.7)
val_df = data_df[data_df['country_code']==test_country]

trn_df = trn_df[trn_df['crop_prob']>0.7]

features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats + ['valid_date_doy','valid_peak_ind','valid_date_ind','n_peaks']

X_trnval_df = trn_df[features_list]
y_trnval_df = trn_df[label_col]
X_tst_df = val_df[features_list]
y_tst_df = val_df[label_col]

class_weights = {'False':1, 'True':1}

model = CatBoostClassifier(
    iterations=500, 
    depth=8,
    eval_metric='F1',
    learning_rate=0.05,
    l2_leaf_reg=130,
    verbose=250,
    random_seed=42,
    class_weights=class_weights
    )

model.fit(X_trnval_df, y_trnval_df)
probs = model.predict_proba(X_tst_df)
pred = model.predict(X_tst_df).flatten()
if y_tst_df.nunique()==2:
    pred = np.array([xx=='True' for xx in pred])

# create dataframe with predictions and useful attributes
preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
preds_df.columns = ['true','pred']
preds_df.set_index(val_df.index, inplace=True)
gc.collect()

print(f1_score(preds_df['true'], preds_df['pred']))

In [ ]:
trn_df = data_df[data_df['country_code']!=test_country]
val_df = data_df[data_df['country_code']==test_country]
test_injection_df = val_df[val_df[label_col]].sample(n=n_test_samples)

In [ ]:
test_country = 'UKR'
tlist = [1,10,20,30,40,50,100,200,250,300,350,400,450,500,550,600,650,700,750,800,900,1000,1100,1200]

tres = []
pbar = tqdm(total=len(tlist))
for n_test_samples in tlist:
    pbar.update(1)
    trn_df = data_df[data_df['country_code']!=test_country]
    val_df = data_df[data_df['country_code']==test_country]

    if ((val_df[label_col]).sum()<n_test_samples) | ((~val_df[label_col]).sum()<n_test_samples): continue

    test_injection_df = val_df[val_df[label_col]].sample(n=n_test_samples)
    trn_df = pd.concat((trn_df,test_injection_df), axis=0)
    trn_df = pd.concat((trn_df,val_df[~val_df[label_col]].sample(n=n_test_samples)), axis=0)

    trn_df = trn_df[trn_df['crop_prob']>0.7]

    features_list = optical12_feats + sar12_feats + temp12_feats + prcp12_feats + dem_feats + latlon_feats + ndvi_feats + ['valid_date_ind']
    X_trnval_df = trn_df[features_list]
    y_trnval_df = trn_df[label_col]
    X_tst_df = val_df[features_list]
    y_tst_df = val_df[label_col]

    model = CatBoostClassifier(
        iterations=500, 
        depth=8,
        eval_metric='F1',
        learning_rate=0.05,
        l2_leaf_reg=130,
        verbose=0,
        random_seed=42
        )

    model.fit(X_trnval_df, y_trnval_df)
    pred = model.predict(X_tst_df).flatten()
    if y_tst_df.nunique()==2:
        pred = np.array([xx=='True' for xx in pred])

    # create dataframe with predictions and useful attributes
    preds_df = pd.DataFrame([y_tst_df.values, pred]).transpose()
    preds_df.columns = ['true','pred']
    _f1 = f1_score(preds_df['true'], preds_df['pred'])
    tres.append([n_test_samples, _f1])

In [ ]:
pd.DataFrame(tres)

In [ ]:
print(classification_report(
    preds_df['true'], 
    preds_df['pred'], 
    target_names=['not_maize','maize'])) 

In [ ]:
print(classification_report(
    preds_df[preds_df['crop_prob']>0.7]['true'], 
    preds_df[preds_df['crop_prob']>0.7]['pred'], 
    target_names=['not_maize','maize'])) 

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['country_code','n_peaks'], ascending=True).iloc[:20]

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['country_code','n_peaks'], ascending=True).iloc[:20]

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['n_maize_pixels'], ascending=False)

In [ ]:
tres[tres['n_maize_pixels']>10].sort_values(by=['n_maize_pixels'], ascending=False)

In [ ]:
label_columns

In [ ]:
label_columns.remove('cropland')
label_columns.remove('cropgroup')

In [ ]:
# create train/test datasets
stratification_col = 'croptype'
test_size = 0.3

trnval_df, tst_df = train_test_split(
    data_df,
    stratify=data_df[stratification_col],
    test_size=test_size,
    random_state=42)

X_trn_df = trnval_df[features_list]
y_trn_df = trnval_df[label_columns].map(tmap)

X_tst_df = tst_df[features_list]
y_tst_df = tst_df[label_columns].map(tmap)

In [ ]:
cb_clsfr = CatBoostClassifierWrapper(
    iterations=2000, 
    depth=8,
    eval_metric='F1',
    learning_rate=0.3,
    l2_leaf_reg=100,
    verbose=50,
    random_seed=42,
)

classifier = LocalClassifierPerNode(
    local_classifier=cb_clsfr,
    n_jobs=-1,
)

classifier.fit(X_trn_df, y_trn_df)
# pred, probs = classifier.predict_proba(X_tst_df)
pred = classifier.predict(X_tst_df)
# pred = pred.astype('int32')

In [ ]:
# assess predictions at a level
level_ind = -1

level_true = y_tst_df[label_columns[level_ind]]
level_pred = pred[:,level_ind]

# map to human names
tmap = ec_map[['{}_label'.format(label_columns[level_ind]),'{}_name'.format(label_columns[level_ind])]].drop_duplicates()
tmap = dict(tmap.to_numpy())
level_true = level_true.map(tmap)
level_pred = [*map(tmap.get, level_pred)]

In [ ]:
print(classification_report(
    level_true, 
    level_pred, 
    target_names=np.sort(level_true.unique()))) 